# FIFA World Cup 2026 — Difficulty of Path to the Final

Three independent metrics, ranked separately:
1. **ELO strength** — average World Football ELO rating of opponents on the path to the final
2. **Win probability product** — product of per-match win probabilities (lower = harder path)
3. **FIFA ranking** — average FIFA ranking of opponents (lower rank number = stronger opponent = harder)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_rows', 60)

## 1. Team Ratings

Update these dicts with current values before running:
- ELO: https://www.eloratings.net
- FIFA rankings: https://inside.fifa.com/fifa-world-ranking/men

In [ ]:
elo_ratings = {
    'Argentina': 2141, 'France': 2005, 'Spain': 2033, 'Brazil': 2083,
    'England': 1985, 'Portugal': 2049, 'Germany': 1988, 'Netherlands': 1970,
    'Belgium': 1946, 'Italy': 1946, 'Croatia': 1931, 'Denmark': 1934,
    'Uruguay': 1916, 'USA': 1893, 'Mexico': 1883, 'Colombia': 1900,
    'Japan': 1887, 'Switzerland': 1890, 'Morocco': 1827, 'Senegal': 1836,
    'South Korea': 1836, 'Chile': 1842, 'Ecuador': 1816, 'Australia': 1795,
    'Serbia': 1868, 'Canada': 1772, 'Nigeria': 1768, 'Peru': 1786,
    'Poland': 1832, 'Iran': 1756, 'Ivory Coast': 1792, 'Ghana': 1730,
    'Egypt': 1716, 'Tunisia': 1737, 'Cameroon': 1762, 'South Africa': 1662,
    'Paraguay': 1720, 'Saudi Arabia': 1703, 'Bolivia': 1698, 'Costa Rica': 1762,
    'DR Congo': 1690, 'Qatar': 1639,
}

fifa_rankings = {
    'Argentina': 1, 'France': 2, 'Spain': 3, 'England': 4,
    'Brazil': 5, 'Portugal': 6, 'Belgium': 7, 'Netherlands': 8,
    'Italy': 9, 'Germany': 10, 'Colombia': 12, 'Morocco': 14,
    'Japan': 15, 'USA': 16, 'Mexico': 17, 'Uruguay': 18,
    'Denmark': 19, 'Senegal': 20, 'Switzerland': 21, 'South Korea': 22,
    'Australia': 23, 'Poland': 26, 'Tunisia': 30, 'Egypt': 35,
    'Qatar': 38, 'Canada': 40, 'Cameroon': 43, 'Ecuador': 45,
    'Costa Rica': 49, 'Nigeria': 50, 'Chile': 55, 'Saudi Arabia': 56,
    'DR Congo': 58, 'Peru': 70, 'South Africa': 72, 'Ghana': 66,
    'Serbia': 33, 'Croatia': 11, 'Iran': 21, 'Ivory Coast': 60,
    'Paraguay': 62, 'Bolivia': 80,
}

print(f"ELO data: {len(elo_ratings)} teams | FIFA ranking data: {len(fifa_rankings)} teams")

## 2. 2026 World Cup Groups

48 teams, 12 groups of 4. Top 2 per group + 8 best 3rd-place teams advance to the Round of 32.

In [ ]:
# Update with the official draw once confirmed
groups = {
    'A': ['Mexico',      'Argentina',   'Canada',      'Ecuador'],
    'B': ['USA',         'Brazil',      'Colombia',    'Peru'],
    'C': ['France',      'Morocco',     'Ivory Coast', 'Belgium'],
    'D': ['Spain',       'Germany',     'Japan',       'South Korea'],
    'E': ['England',     'Portugal',    'Iran',        'Australia'],
    'F': ['Netherlands', 'Italy',       'Egypt',       'South Africa'],
    'G': ['Croatia',     'Uruguay',     'Senegal',     'Bolivia'],
    'H': ['Denmark',     'Switzerland', 'Serbia',      'Cameroon'],
    'I': ['Belgium',     'Chile',       'Nigeria',     'DR Congo'],
    'J': ['Poland',      'Costa Rica',  'Paraguay',    'Saudi Arabia'],
    'K': ['Canada',      'Ghana',       'Tunisia',     'Qatar'],
    'L': ['South Korea', 'USA',         'Australia',   'Iran'],
}

rows = []
for grp, teams in groups.items():
    for team in teams:
        rows.append({
            'group': grp, 'team': team,
            'elo': elo_ratings.get(team, np.nan),
            'fifa_rank': fifa_rankings.get(team, np.nan),
        })

df_teams = pd.DataFrame(rows).drop_duplicates('team').reset_index(drop=True)
df_teams.sort_values('elo', ascending=False).head(20)

## 3. Core Functions

In [ ]:
def elo_win_prob(elo_a: float, elo_b: float) -> float:
    """P(A beats B) using standard ELO formula."""
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))


def score_path(team: str, opponents: list[str]) -> dict:
    """Return all three difficulty metrics for a given path."""
    team_elo = elo_ratings.get(team, np.nan)
    opp_elos  = [elo_ratings.get(o, np.nan) for o in opponents]
    opp_ranks = [fifa_rankings.get(o, np.nan) for o in opponents]

    win_probs = [elo_win_prob(team_elo, e) for e in opp_elos if not np.isnan(e)]

    return {
        'team':              team,
        'opponents':         opponents,
        'avg_opp_elo':       float(np.nanmean(opp_elos)),
        'avg_opp_fifa_rank': float(np.nanmean(opp_ranks)),
        'win_prob_product':  float(np.prod(win_probs)) if win_probs else np.nan,
    }


# Sanity check
ex = score_path('Argentina', ['France', 'Brazil', 'Spain', 'Germany'])
print(f"Argentina path through [France, Brazil, Spain, Germany]")
print(f"  Avg opp ELO:        {ex['avg_opp_elo']:.0f}")
print(f"  Avg opp FIFA rank:  {ex['avg_opp_fifa_rank']:.1f}")
print(f"  Win prob to final:  {ex['win_prob_product']*100:.2f}%")

## 4. Monte Carlo Group Stage Simulation

10,000 simulations per group to get each team's probability of finishing 1st / 2nd / 3rd.

In [ ]:
import itertools
from collections import defaultdict

DRAW_RATE = 0.28  # ~28% of World Cup group matches end in draws
N_SIMS = 10_000


def simulate_group(teams: list[str], n_sims: int = N_SIMS) -> dict[str, list[float]]:
    """Return P(finish position k) for each team. Index 0 = 1st place."""
    counts = {t: [0] * len(teams) for t in teams}

    for _ in range(n_sims):
        pts = {t: 0 for t in teams}
        gd  = {t: 0 for t in teams}  # goal-diff tiebreaker (ELO proxy)
        for a, b in itertools.combinations(teams, 2):
            ea, eb = elo_ratings.get(a, 1500), elo_ratings.get(b, 1500)
            p_a_win = elo_win_prob(ea, eb) * (1 - DRAW_RATE)
            p_draw  = DRAW_RATE
            r = np.random.random()
            if r < p_a_win:
                pts[a] += 3; gd[a] += 1; gd[b] -= 1
            elif r < p_a_win + p_draw:
                pts[a] += 1; pts[b] += 1
            else:
                pts[b] += 3; gd[b] += 1; gd[a] -= 1

        ranked = sorted(teams, key=lambda t: (pts[t], gd[t], elo_ratings.get(t, 0)), reverse=True)
        for pos, team in enumerate(ranked):
            counts[team][pos] += 1

    return {t: [counts[t][i] / n_sims for i in range(len(teams))] for t in teams}


all_group_probs = {grp: simulate_group(teams) for grp, teams in groups.items()}

# Show Group A as example
print("Group A — finish probabilities (1st / 2nd / 3rd / 4th):")
for team in groups['A']:
    p = all_group_probs['A'][team]
    print(f"  {team:<20} {p[0]:.1%}  {p[1]:.1%}  {p[2]:.1%}  {p[3]:.1%}")

## 5. Build Expected Knockout Paths

For each likely qualifier, build the expected path to the final using ELO-weighted opponent probabilities.

In [ ]:
def expected_qualifiers(n_per_group: int = 2) -> list[tuple[str, str, int]]:
    """Return (team, group, finish_position) for likely qualifiers."""
    result = []
    for grp, teams in groups.items():
        ranked = sorted(teams,
                        key=lambda t: all_group_probs[grp][t][0],
                        reverse=True)
        for pos, team in enumerate(ranked[:n_per_group]):
            result.append((team, grp, pos + 1))
    return result


qualifiers = expected_qualifiers()
qualifier_teams = [t for t, _, _ in qualifiers]
print(f"{len(qualifier_teams)} qualifiers across {len(groups)} groups")


def build_path_opponents(team: str, team_group: str, n_rounds: int = 5) -> list[str]:
    """
    Estimate opponents at each knockout round.
    Uses ELO-weighted expected opponent from the pool of qualifiers in the
    opposing bracket half. Rounds: R32, R16, QF, SF, Final.
    """
    # Opponents = qualifiers not in same group, sorted strongest first
    pool = [
        t for t, g, _ in qualifiers
        if t != team and g != team_group
    ]
    pool.sort(key=lambda t: elo_ratings.get(t, 0), reverse=True)
    # Distribute: face progressively stronger opponents as rounds advance
    # Split pool into thirds: easy side of draw, medium, hard (SF, Final)
    n = len(pool)
    weak   = pool[n//2:]          # weaker half (R32, R16)
    strong = pool[:n//2]          # stronger half (QF onward)
    rounds = [
        weak[-1]   if weak   else pool[0],   # R32 — weakest expected
        weak[0]    if weak   else pool[1],   # R16
        strong[-1] if strong else pool[2],   # QF
        strong[1]  if len(strong) > 1 else pool[3],  # SF
        strong[0]  if strong else pool[4],   # Final
    ]
    return rounds[:n_rounds]


records = []
for team, grp, finish in qualifiers:
    opponents = build_path_opponents(team, grp)
    metrics = score_path(team, opponents)
    metrics['group'] = grp
    metrics['group_finish'] = finish
    records.append(metrics)

df = pd.DataFrame(records)[['team', 'group', 'group_finish',
                              'avg_opp_elo', 'avg_opp_fifa_rank', 'win_prob_product']]
df.head()

## 6. Rank by Each Metric

In [ ]:
df['rank_elo']      = df['avg_opp_elo'].rank(ascending=False).astype(int)
df['rank_fifa']     = df['avg_opp_fifa_rank'].rank(ascending=True).astype(int)   # lower FIFA rank = harder
df['rank_win_prob'] = df['win_prob_product'].rank(ascending=True).astype(int)    # lower prob = harder
df['composite']     = ((df['rank_elo'] + df['rank_fifa'] + df['rank_win_prob']) / 3).round(2)

df_sorted = df.sort_values('composite').reset_index(drop=True)
df_sorted.index += 1

print("Hardest paths to the final (composite of all 3 metrics):")
df_sorted[['team', 'group', 'avg_opp_elo', 'avg_opp_fifa_rank',
           'win_prob_product', 'rank_elo', 'rank_fifa', 'rank_win_prob', 'composite']]

## 7. Visualisations

In [ ]:
top = df_sorted.head(15)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('FIFA World Cup 2026 — Hardest Paths to the Final', fontsize=14, fontweight='bold')

axes[0].barh(top['team'][::-1], top['avg_opp_elo'][::-1], color='steelblue')
axes[0].set_title('Metric 1: Avg Opponent ELO')
axes[0].set_xlabel('ELO (higher = harder)')

axes[1].barh(top['team'][::-1], top['avg_opp_fifa_rank'][::-1], color='coral')
axes[1].set_title('Metric 2: Avg Opponent FIFA Rank')
axes[1].set_xlabel('Rank number (lower = harder)')

axes[2].barh(top['team'][::-1], top['win_prob_product'][::-1], color='mediumseagreen')
axes[2].set_title('Metric 3: Win Prob to Reach Final')
axes[2].set_xlabel('Probability (lower = harder)')

plt.tight_layout()
plt.savefig('path_difficulty_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

sc = ax.scatter(
    df['avg_opp_elo'],
    df['win_prob_product'],
    s=300 / (df['avg_opp_fifa_rank'] + 1) * 30,
    c=df['composite'],
    cmap='RdYlGn_r',
    alpha=0.75,
    edgecolors='grey',
    linewidths=0.5,
)

for _, row in df.iterrows():
    ax.annotate(row['team'],
                xy=(row['avg_opp_elo'], row['win_prob_product']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=8)

plt.colorbar(sc, ax=ax, label='Composite difficulty rank (1 = hardest path)')
ax.set_xlabel('Avg Opponent ELO  →  harder right')
ax.set_ylabel('Win probability to reach final  →  harder down')
ax.set_title('Path Difficulty Scatter\n(bubble size ∝ opponent FIFA ranking strength)')
plt.tight_layout()
plt.savefig('path_difficulty_scatter.png', dpi=150, bbox_inches='tight')
plt.show()